In [0]:
# This notebook makes predictions on live stations using either the champion or challenger models

from datetime import datetime, timedelta, timezone

import pandas as pd
import mlflow
from mlflow import MlflowClient
from pyspark.sql import functions as F, Window

from station_features import build_predictors



# Scheduled jobs pass run_mode = "production". Interactive runs default to "dev".
dbutils.widgets.dropdown("run_mode", "dev", ["dev", "production"])
RUN_MODE = dbutils.widgets.get("run_mode")
IS_PRODUCTION = RUN_MODE == "production"

spark.conf.set("spark.sql.session.timeZone", "UTC")

# Production predicts from the live poller's tables. Dev uses the downloaded silver tables for tests.
STATUS_TABLE = ("workspace.default.live_station_status" if IS_PRODUCTION
                else "workspace.default.silver_station_status")
INFO_TABLE = ("workspace.default.live_station_info" if IS_PRODUCTION
              else "workspace.default.silver_station_info")

MODEL_NAMES = {"bike": "workspace.default.bike_stockout_lr", "dock": "workspace.default.dock_stockout_lr"}


PREDICTION_TIME = None # UTC timestamp string to score as of, e.g. "2026-09-20T12:00:00"; None = newest poll


# Dev runs write to a separate table, so tests never mix with live predictions.
PREDICTIONS_TABLE = ("workspace.default.station_predictions" if IS_PRODUCTION
                     else "workspace.default.station_predictions_dev")

# A production run always scores the newest poll. A fixed time is only for tests.
if IS_PRODUCTION and PREDICTION_TIME is not None:
    raise ValueError("PREDICTION_TIME must be None in production.")

# We can setup the model to score the challenger model on stations
MODEL_ALIAS = "champion" if IS_PRODUCTION else "challenger"

HISTORY_MINUTES = 45     # 30 minutes of rate history, 10 minutes of gap allowance, a little margin
MAX_STALENESS_MINUTES = 10   # ignore stations whose latest data is older than this
HORIZON_MIN = 30
LOCAL_TZ = "America/New_York"
mlflow.set_registry_uri("databricks-uc")

In [0]:
# 1. Recent status and the information snapshots that apply to it.

station_status = spark.table(STATUS_TABLE)

prediction_time = (datetime.fromisoformat(PREDICTION_TIME) if PREDICTION_TIME
         else station_status.agg(F.max("fetched_at")).first()[0])

history_start_time = prediction_time - timedelta(minutes=HISTORY_MINUTES)

print("Predictions made for", prediction_time, "UTC")

# We filter to station status updates that are more recent that history_start_time
# And where the retrieval time is before the prediction_time

recent_status = station_status.filter(
    (F.col("fetched_at") > F.lit(history_start_time)) & (F.col("fetched_at") <= F.lit(prediction_time))
).select("station_id", "fetched_at", "num_bikes_available", "num_docks_available",
         "is_installed", "is_renting", "is_returning")

# We get the latest station_info
station_info = (spark.table(INFO_TABLE)
        .filter(F.col("fetched_at") <= F.lit(prediction_time))
        .select("station_id", "fetched_at", "name", "capacity", "lat", "lon").distinct())

# 2. Generate features and acquire latest station info
predictors = build_predictors(recent_status, station_info,local_timezone=LOCAL_TZ)

# Finds the latest row for each station
newest_rows = Window.partitionBy("station_id").orderBy(F.col("fetched_at").desc())

# Then we generate the current features
# This is one row for per station

current_features_spark = (predictors
           .withColumn("_rank", F.row_number().over(newest_rows)).filter("_rank = 1").drop("_rank")
           .filter(F.col("fetched_at") >= F.lit(prediction_time - timedelta(minutes=MAX_STALENESS_MINUTES))))

columns = ["station_id", "fetched_at", "station_name", "lat", "lon", "geographic_cell",
           "log_capacity", "weekend", "hour_sin_1", "hour_cos_1", "hour_sin_2", "hour_cos_2",
           "bike_stockout", "num_bikes_available", "bike_available_fraction",
           "bike_change_30min_per_minute_fraction",
           "dock_stockout", "num_docks_available", "dock_available_fraction",
           "dock_change_30min_per_minute_fraction"]
station_features = current_features_spark.select(*columns).toPandas()
station_features.attrs = {}
print(f"{len(station_features):,} stations with a fresh poll")

In [0]:
# 3. Predict each outcome for the stations eligible for it.
client = MlflowClient(registry_uri="databricks-uc")
prediction_at = datetime.now(timezone.utc).replace(tzinfo=None)
model_version = {}

# We will loop over the two types of predictions

for target in ["bike", "dock"]:
    model_name = MODEL_NAMES[target]
    # Get the version number of the current model
    model_version[target] = client.get_model_version_by_alias(model_name, MODEL_ALIAS).version
    # Get the model from the registry
    model = mlflow.pyfunc.load_model(f"models:/{model_name}@{MODEL_ALIAS}")

    # Current state: eligible (operating, not stocked out), stocked_out, or unknown (not operating).
    current_state = station_features[f"{target}_stockout"]

    # We will only make predictions for stations that are currently not experiencing a stockout

    station_features[f"{target}_state"] = pd.Series(
        ["unknown"] * len(station_features), index=station_features.index
    ).where(current_state.isna(), current_state.map({False: "eligible", True: "stocked_out"}))

    eligible_stations_mask = station_features[f"{target}_state"] == "eligible"

    # We filter to the stations that need a prediction:
    predictors = station_features.loc[eligible_stations_mask, ["geographic_cell", "log_capacity", "weekend",
                                     "hour_sin_1", "hour_cos_1", "hour_sin_2", "hour_cos_2",
                                     f"num_{target}s_available", f"{target}_available_fraction",
                                     f"{target}_change_30min_per_minute_fraction"]]
    # The model's signature expects floats; the counts arrive from Spark as integers
    predictors = predictors.astype({c: "float64" for c in predictors.columns if c != "geographic_cell"})
    station_features[f"p_{target}_stockout"] = float("nan")

    # This predicts the probability, the predict method is the predict_proba saved when we registered the model
    if len(predictors):
        station_features.loc[eligible_stations_mask, f"p_{target}_stockout"] = model.predict(predictors)[:, 1]

    print(f"{target}: {int(eligible_stations_mask.sum()):,} eligible stations scored with version {model_version[target]}")

predictions = station_features[["station_id", "fetched_at", "station_name", "lat", "lon",
                        "num_bikes_available", "bike_state", "p_bike_stockout",
                        "num_docks_available", "dock_state", "p_dock_stockout"]].copy()
predictions["prediction_at"] = prediction_at
predictions["horizon_minutes"] = HORIZON_MIN
predictions["bike_model_version"] = str(model_version["bike"])
predictions["dock_model_version"] = str(model_version["dock"])
predictions["prediction_time"] = prediction_time

In [0]:
# 4. Save the predictions, but only once for each prediction time.
new_predictions = spark.createDataFrame(predictions)

# Stations without a prediction get null, not NaN, so averages and dashboards ignore them
probability_columns = ["p_bike_stockout", "p_dock_stockout"]
new_predictions = new_predictions.withColumns(
    {column: F.when(~F.isnan(column), F.col(column)) for column in probability_columns})

if spark.catalog.tableExists(PREDICTIONS_TABLE):
    # A rerun for the same poll must not add a second copy of its predictions
    already_scored = not (spark.table(PREDICTIONS_TABLE)
                          .filter(F.col("prediction_time") == F.lit(prediction_time))
                          .isEmpty())
    if already_scored:
        print(f"Predictions for {prediction_time} already exist; nothing written.")
    else:
        new_predictions.write.mode("append").saveAsTable(PREDICTIONS_TABLE)
        print(f"{len(predictions):,} rows appended to {PREDICTIONS_TABLE}")
else:
    new_predictions.write.saveAsTable(PREDICTIONS_TABLE)
    print(f"Created {PREDICTIONS_TABLE} with {len(predictions):,} rows")

# The stations with the highest risk at the prediction time
display(
    new_predictions.filter("bike_state = 'eligible'")
    .select("station_name", "num_bikes_available", "p_bike_stockout")
    .orderBy(F.desc("p_bike_stockout")).limit(15)
)
display(
    new_predictions.filter("dock_state = 'eligible'")
    .select("station_name", "num_docks_available", "p_dock_stockout")
    .orderBy(F.desc("p_dock_stockout")).limit(15)
)